# 🚕 NYC Taxi Trip Analytics using Apache Spark
## Big Data Analytics (DS-313) — Assignment 3
**Student Name:** Muhammad Hamza Azeem
**Roll Number:** 221980023
**Date:** 1st August 2026
**Environment:** Google Colab + PySpark

In [1]:
# STEP 0: INSTALL PYSPARK & DOWNLOAD DATASET
# Run this cell FIRST — it installs PySpark and downloads
# the NYC Yellow Taxi dataset (January 2024)

!pip install pyspark==3.5.5 -q

import os
os.makedirs("output", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

# Download the dataset directly from NYC TLC
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
print("✅ PySpark installed")
print("✅ Dataset downloaded: yellow_tripdata_2024-01.parquet")
print(f"✅ File size: {os.path.getsize('yellow_tripdata_2024-01.parquet') / (1024*1024):.1f} MB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 MB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.5 which is incompatible.
✅ PySpark installed
✅ Dataset downloaded: yellow_tripdata_2024-01.parquet
✅ File size: 47.6 MB


---
## Part 1: Environment Setup (10 Marks)


In [2]:

import sys
import os
import platform
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

# Create Spark Session
spark = SparkSession.builder \
    .appName("NYC Taxi Trip Analytics") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

print("=" * 60)
print("        ENVIRONMENT SETUP — SYSTEM INFORMATION")
print("=" * 60)
print(f"  Python Version     : {sys.version.split()[0]}")
print(f"  Spark Version      : {spark.version}")
print(f"  Operating System   : {platform.system()} {platform.release()}")
print(f"  Architecture       : {platform.machine()}")
print(f"  Java Home          : {os.environ.get('JAVA_HOME', 'Not set')}")
print(f"  Spark Master       : {spark.sparkContext.master}")
print(f"  App Name           : {spark.sparkContext.appName}")
print(f"  Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"  Environment        : Google Colab")
print("=" * 60)
print("\n📋 Spark Configuration:")
for key, value in sorted(spark.sparkContext.getConf().getAll()):
    print(f"    {key} = {value}")
print(f"\n🌐 Spark UI: http://localhost:4040 (accessible via Colab proxy)")

# Check Java version
print("\n📋 Java Version:")
!java -version

        ENVIRONMENT SETUP — SYSTEM INFORMATION
  Python Version     : 3.12.13
  Spark Version      : 3.5.5
  Operating System   : Linux 6.6.122+
  Architecture       : x86_64
  Java Home          : Not set
  Spark Master       : local[*]
  App Name           : NYC Taxi Trip Analytics
  Default Parallelism: 2
  Environment        : Google Colab

📋 Spark Configuration:
    spark.app.id = local-1785573925730
    spark.app.name = NYC Taxi Trip Analytics
    spark.app.startTime = 1785573924279
    spark.app.submitTime = 1785573924020
    spark.driver.extraJavaOptions = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurre

---
## Part 2: Load Dataset (10 Marks)


In [3]:

df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

print("=" * 60)
print("        DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("\n📋 SCHEMA:")
df.printSchema()

record_count = df.count()
print(f"\n📊 Number of Columns : {len(df.columns)}")
print(f"📊 Number of Records : {record_count:,}")
print(f"\n📋 Column Names: {df.columns}")

        DATASET LOADED SUCCESSFULLY

📋 SCHEMA:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)


📊 Number of Columns : 19
📊 Number of Records : 2,964,624

📋 Column Names: ['V

In [4]:
print("📋 FIRST 20 ROWS:")
df.show(20, truncate=False)

📋 FIRST 20 ROWS:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |

---
## Part 3: Exploratory Data Analysis (15 Marks)


In [5]:

print("=" * 60)
print("        EXPLORATORY DATA ANALYSIS")
print("=" * 60)

total_trips = df.count()
earliest_date = df.select(min("tpep_pickup_datetime")).collect()[0][0]
latest_date = df.select(max("tpep_pickup_datetime")).collect()[0][0]
unique_vendors = df.select("VendorID").distinct().count()
avg_distance = df.select(avg("trip_distance")).collect()[0][0]
avg_fare = df.select(avg("fare_amount")).collect()[0][0]
max_fare = df.select(max("fare_amount")).collect()[0][0]
min_fare = df.select(min("fare_amount")).collect()[0][0]
avg_passengers = df.select(avg("passenger_count")).collect()[0][0]
payment_methods = df.select("payment_type").distinct().count()

print(f"\n  1.  Total Trips             : {total_trips:,}")
print(f"  2.  Earliest Trip Date      : {earliest_date}")
print(f"  3.  Latest Trip Date        : {latest_date}")
print(f"  4.  Unique Vendors          : {unique_vendors}")
print(f"  5.  Average Trip Distance   : {avg_distance:.2f} miles")
print(f"  6.  Average Fare            : ${avg_fare:.2f}")
print(f"  7.  Maximum Fare            : ${max_fare:.2f}")
print(f"  8.  Minimum Fare            : ${min_fare:.2f}")
print(f"  9.  Average Passenger Count : {avg_passengers:.2f}")
print(f"  10. Number of Payment Methods: {payment_methods}")
print("\n" + "=" * 60)

        EXPLORATORY DATA ANALYSIS

  1.  Total Trips             : 2,964,624
  2.  Earliest Trip Date      : 2002-12-31 22:59:39
  3.  Latest Trip Date        : 2024-02-01 00:01:15
  4.  Unique Vendors          : 3
  5.  Average Trip Distance   : 3.65 miles
  6.  Average Fare            : $18.18
  7.  Maximum Fare            : $5000.00
  8.  Minimum Fare            : $-899.00
  9.  Average Passenger Count : 1.34
  10. Number of Payment Methods: 5



---
## Part 4: Data Cleaning (10 Marks)


In [6]:

print("=" * 60)
print("        DATA CLEANING")
print("=" * 60)

original_count = df.count()
print(f"\n🔴 Before Cleaning: {original_count:,} records")

# Step 1: Remove Duplicates
# Duplicates can skew analysis results and inflate trip counts
df_clean = df.dropDuplicates()
after_dedup = df_clean.count()
print(f"\n✅ Step 1 — Remove Duplicates")
print(f"   Removed: {original_count - after_dedup:,} duplicate rows")
print(f"   Remaining: {after_dedup:,} records")

# Step 2: Remove Invalid Trip Distances (zero or negative)
# Trips with 0 or negative distance are data errors
before = df_clean.count()
df_clean = df_clean.filter(col("trip_distance") > 0)
after = df_clean.count()
print(f"\n✅ Step 2 — Remove Invalid Trip Distances (distance <= 0)")
print(f"   Removed: {before - after:,} invalid rows")
print(f"   Remaining: {after:,} records")

# Step 3: Remove Negative Fares
# Negative fares are billing errors or refunds
before = df_clean.count()
df_clean = df_clean.filter(col("fare_amount") >= 0)
after = df_clean.count()
print(f"\n✅ Step 3 — Remove Negative Fares")
print(f"   Removed: {before - after:,} invalid rows")
print(f"   Remaining: {after:,} records")

# Step 4: Handle Missing Values
# Drop rows where critical columns have null values
before = df_clean.count()
critical_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
                 "passenger_count", "trip_distance", "fare_amount", "total_amount"]
df_clean = df_clean.na.drop(subset=critical_cols)
after = df_clean.count()
print(f"\n✅ Step 4 — Handle Missing Values (drop nulls in critical columns)")
print(f"   Removed: {before - after:,} rows with missing values")
print(f"   Remaining: {after:,} records")

print(f"\n{'=' * 60}")
print(f"🟢 After Cleaning: {df_clean.count():,} records")
print(f"   Total Removed : {original_count - df_clean.count():,} records")
print(f"   Retention Rate: {df_clean.count()/original_count*100:.1f}%")
print(f"{'=' * 60}")

        DATA CLEANING

🔴 Before Cleaning: 2,964,624 records

✅ Step 1 — Remove Duplicates
   Removed: 0 duplicate rows
   Remaining: 2,964,624 records

✅ Step 2 — Remove Invalid Trip Distances (distance <= 0)
   Removed: 60,371 invalid rows
   Remaining: 2,904,253 records

✅ Step 3 — Remove Negative Fares
   Removed: 34,065 invalid rows
   Remaining: 2,870,188 records

✅ Step 4 — Handle Missing Values (drop nulls in critical columns)
   Removed: 115,293 rows with missing values
   Remaining: 2,754,895 records

🟢 After Cleaning: 2,754,895 records
   Total Removed : 209,729 records
   Retention Rate: 92.9%


---
## Part 5: Spark Transformations (15 Marks)
Using 10 transformations: filter, select, withColumn, orderBy, drop, distinct, groupBy, join, alias, repartition



In [7]:
# ──── Transformation 1: filter() ────
# Purpose: Filter trips with distance greater than 5 miles
print("📌 Transformation 1: filter()")
print("   Purpose: Select trips longer than 5 miles")
df_filtered = df_clean.filter(col("trip_distance") > 5)
print(f"   Long trips (>5 miles): {df_filtered.count():,}")
df_filtered.select("VendorID", "trip_distance", "fare_amount", "total_amount").show(5)

📌 Transformation 1: filter()
   Purpose: Select trips longer than 5 miles
   Long trips (>5 miles): 433,286
+--------+-------------+-----------+------------+
|VendorID|trip_distance|fare_amount|total_amount|
+--------+-------------+-----------+------------+
|       1|          6.2|       32.4|        39.4|
|       2|         5.96|       31.7|       45.88|
|       2|         5.68|       28.9|       42.37|
|       1|          8.2|       38.7|        48.7|
|       2|        10.51|       44.3|       56.24|
+--------+-------------+-----------+------------+
only showing top 5 rows



In [8]:
# ──── Transformation 2: select() ────
# Purpose: Select only relevant columns for focused analysis
print("📌 Transformation 2: select()")
print("   Purpose: Select key columns for analysis")
df_selected = df_clean.select("VendorID", "tpep_pickup_datetime",
                               "trip_distance", "fare_amount",
                               "tip_amount", "total_amount", "payment_type")
df_selected.show(5)

📌 Transformation 2: select()
   Purpose: Select key columns for analysis
+--------+--------------------+-------------+-----------+----------+------------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|tip_amount|total_amount|payment_type|
+--------+--------------------+-------------+-----------+----------+------------+------------+
|       1| 2024-01-01 00:54:08|          4.7|       29.6|       6.9|        41.5|           1|
|       2| 2024-01-01 00:28:08|         0.04|        3.0|       0.0|         8.0|           2|
|       1| 2024-01-01 00:41:06|          1.5|       12.8|      4.45|       22.25|           1|
|       2| 2024-01-01 00:32:34|         2.57|       17.7|      10.0|        32.7|           1|
|       2| 2024-01-01 00:44:01|         2.22|       13.5|       0.0|        18.5|           1|
+--------+--------------------+-------------+-----------+----------+------------+------------+
only showing top 5 rows



In [9]:
# ──── Transformation 3: withColumn() ────
# Purpose: Create new calculated columns — trip duration and cost per mile
print("📌 Transformation 3: withColumn()")
print("   Purpose: Add trip_duration_min and cost_per_mile columns")
df_enriched = df_clean \
    .withColumn("trip_duration_min",
                round((unix_timestamp("tpep_dropoff_datetime") -
                       unix_timestamp("tpep_pickup_datetime")) / 60, 2)) \
    .withColumn("cost_per_mile",
                round(col("fare_amount") / col("trip_distance"), 2))
df_enriched.select("trip_distance", "fare_amount", "trip_duration_min", "cost_per_mile").show(5)

📌 Transformation 3: withColumn()
   Purpose: Add trip_duration_min and cost_per_mile columns
+-------------+-----------+-----------------+-------------+
|trip_distance|fare_amount|trip_duration_min|cost_per_mile|
+-------------+-----------+-----------------+-------------+
|          4.7|       29.6|            32.38|          6.3|
|         0.04|        3.0|             1.13|         75.0|
|          1.5|       12.8|             12.6|         8.53|
|         2.57|       17.7|            16.98|         6.89|
|         2.22|       13.5|             10.5|         6.08|
+-------------+-----------+-----------------+-------------+
only showing top 5 rows



In [10]:
# ──── Transformation 4: orderBy() ────
# Purpose: Find the most expensive trips
print("📌 Transformation 4: orderBy()")
print("   Purpose: Sort by total amount (descending) — top 10 most expensive")
df_sorted = df_clean.orderBy(col("total_amount").desc())
df_sorted.select("VendorID", "trip_distance", "fare_amount", "tip_amount", "total_amount").show(10)

📌 Transformation 4: orderBy()
   Purpose: Sort by total amount (descending) — top 10 most expensive
+--------+-------------+-----------+----------+------------+
|VendorID|trip_distance|fare_amount|tip_amount|total_amount|
+--------+-------------+-----------+----------+------------+
|       2|        31.95|     2221.3|       0.0|      2225.3|
|       2|       233.25|     1616.5|       0.0|      1617.5|
|       2|       142.62|      912.3|       0.0|      940.93|
|       2|       157.25|      899.0|       0.0|       900.0|
|       2|         0.21|      820.0|       0.0|       821.0|
|       2|       109.75|      761.1|       0.0|      775.48|
|       2|       119.46|      739.4|       0.0|      771.41|
|       2|       122.47|      749.2|       0.0|      758.89|
|       2|       120.76|      744.3|       0.0|      753.74|
|       2|         0.11|      700.0|       0.0|      715.75|
+--------+-------------+-----------+----------+------------+
only showing top 10 rows



In [11]:
# ──── Transformation 5: drop() ────
# Purpose: Remove unnecessary columns
print("📌 Transformation 5: drop()")
print("   Purpose: Remove store_and_fwd_flag and Airport_fee columns")
df_dropped = df_clean.drop("store_and_fwd_flag", "Airport_fee")
print(f"   Columns BEFORE drop: {len(df_clean.columns)} → {df_clean.columns}")
print(f"   Columns AFTER  drop: {len(df_dropped.columns)} → {df_dropped.columns}")

📌 Transformation 5: drop()
   Purpose: Remove store_and_fwd_flag and Airport_fee columns
   Columns BEFORE drop: 19 → ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']
   Columns AFTER  drop: 17 → ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']


In [12]:
# ──── Transformation 6: distinct() ────
# Purpose: Find unique vendor IDs and payment types
print("📌 Transformation 6: distinct()")
print("   Purpose: Find unique Vendor IDs and Payment Types")
print("\n   Unique Vendor IDs:")
df_clean.select("VendorID").distinct().show()
print("   Unique Payment Types:")
df_clean.select("payment_type").distinct().orderBy("payment_type").show()

📌 Transformation 6: distinct()
   Purpose: Find unique Vendor IDs and Payment Types

   Unique Vendor IDs:
+--------+
|VendorID|
+--------+
|       1|
|       2|
+--------+

   Unique Payment Types:
+------------+
|payment_type|
+------------+
|           1|
|           2|
|           3|
|           4|
+------------+



In [13]:
# ──── Transformation 7: groupBy() ────
# Purpose: Aggregate trips and revenue by payment type
print("📌 Transformation 7: groupBy()")
print("   Purpose: Trip count and average fare grouped by payment type")
df_grouped = df_clean.groupBy("payment_type").agg(
    count("*").alias("trip_count"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    round(avg("tip_amount"), 2).alias("avg_tip")
)
df_grouped.orderBy("payment_type").show()

📌 Transformation 7: groupBy()
   Purpose: Trip count and average fare grouped by payment type
+------------+----------+--------+-------------+-------+
|payment_type|trip_count|avg_fare|total_revenue|avg_tip|
+------------+----------+--------+-------------+-------+
|           1|   2298422|   18.38|6.452952225E7|   4.16|
|           2|    422945|   18.61|1.008932501E7|    0.0|
|           3|     10649|   17.04|    235562.14|    0.0|
|           4|     22879|   19.67|    572064.61|    0.0|
+------------+----------+--------+-------------+-------+



In [14]:
# ──── Transformation 8: join() ────
# Purpose: Join trip data with vendor names lookup table
print("📌 Transformation 8: join()")
print("   Purpose: Join with vendor names lookup table")
vendor_data = [(1, "Creative Mobile Technologies"),
               (2, "VeriFone Inc.")]
vendor_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("vendor_name", StringType(), True)
])
vendors_df = spark.createDataFrame(vendor_data, schema=vendor_schema)
df_joined = df_clean.join(vendors_df, "VendorID", "left")
df_joined.select("VendorID", "vendor_name", "trip_distance", "fare_amount").show(5)

📌 Transformation 8: join()
   Purpose: Join with vendor names lookup table
+--------+--------------------+-------------+-----------+
|VendorID|         vendor_name|trip_distance|fare_amount|
+--------+--------------------+-------------+-----------+
|       1|Creative Mobile T...|          4.7|       29.6|
|       2|       VeriFone Inc.|         0.04|        3.0|
|       1|Creative Mobile T...|          1.5|       12.8|
|       2|       VeriFone Inc.|         2.57|       17.7|
|       2|       VeriFone Inc.|         2.22|       13.5|
+--------+--------------------+-------------+-----------+
only showing top 5 rows



In [15]:
# ──── Transformation 9: alias() ────
# Purpose: Rename columns for better readability
print("📌 Transformation 9: alias()")
print("   Purpose: Rename columns for clarity")
df_aliased = df_clean.select(
    col("VendorID").alias("Vendor_ID"),
    col("tpep_pickup_datetime").alias("Pickup_Time"),
    col("trip_distance").alias("Distance_Miles"),
    col("fare_amount").alias("Fare_USD"),
    col("tip_amount").alias("Tip_USD"),
    col("total_amount").alias("Total_USD")
)
df_aliased.show(5)

📌 Transformation 9: alias()
   Purpose: Rename columns for clarity
+---------+-------------------+--------------+--------+-------+---------+
|Vendor_ID|        Pickup_Time|Distance_Miles|Fare_USD|Tip_USD|Total_USD|
+---------+-------------------+--------------+--------+-------+---------+
|        1|2024-01-01 00:54:08|           4.7|    29.6|    6.9|     41.5|
|        2|2024-01-01 00:28:08|          0.04|     3.0|    0.0|      8.0|
|        1|2024-01-01 00:41:06|           1.5|    12.8|   4.45|    22.25|
|        2|2024-01-01 00:32:34|          2.57|    17.7|   10.0|     32.7|
|        2|2024-01-01 00:44:01|          2.22|    13.5|    0.0|     18.5|
+---------+-------------------+--------------+--------+-------+---------+
only showing top 5 rows



In [16]:
# ──── Transformation 10: repartition() ────
# Purpose: Change number of partitions for parallel processing
print("📌 Transformation 10: repartition()")
print("   Purpose: Change number of partitions for parallelism")
print(f"   Partitions BEFORE: {df_clean.rdd.getNumPartitions()}")
df_repartitioned = df_clean.repartition(8)
print(f"   Partitions AFTER : {df_repartitioned.rdd.getNumPartitions()}")

📌 Transformation 10: repartition()
   Purpose: Change number of partitions for parallelism
   Partitions BEFORE: 3
   Partitions AFTER : 8


---
## Part 6: Spark SQL (15 Marks)
Create a temporary view and answer 10 SQL questions



In [17]:
# Create temporary SQL view
df_clean.createOrReplaceTempView("taxi_trips")
print("✅ Temporary view 'taxi_trips' created successfully")

✅ Temporary view 'taxi_trips' created successfully


In [18]:
# ──── SQL Query 1: Top 10 Longest Trips ────
print("📌 SQL Query 1: Top 10 Longest Trips")
q1 = spark.sql("""
    SELECT trip_distance,
           fare_amount,
           total_amount,
           tpep_pickup_datetime,
           tpep_dropoff_datetime
    FROM taxi_trips
    ORDER BY trip_distance DESC
    LIMIT 10
""")
q1.show(truncate=False)

📌 SQL Query 1: Top 10 Longest Trips
+-------------+-----------+------------+--------------------+---------------------+
|trip_distance|fare_amount|total_amount|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------+-----------+------------+--------------------+---------------------+
|15400.32     |28.9       |39.48       |2024-01-17 14:50:40 |2024-01-17 15:22:48  |
|10879.28     |70.0       |98.88       |2024-01-03 11:02:57 |2024-01-03 11:47:35  |
|1715.22      |70.0       |98.88       |2024-01-14 17:22:52 |2024-01-14 18:05:15  |
|971.8        |21.5       |23.0        |2024-01-02 08:13:18 |2024-01-02 09:43:30  |
|964.6        |39.5       |41.0        |2024-01-02 08:40:41 |2024-01-02 08:55:27  |
|277.4        |33.8       |39.55       |2024-01-31 15:31:15 |2024-01-31 15:56:06  |
|246.22       |8.6        |18.12       |2024-01-17 16:13:24 |2024-01-17 16:21:44  |
|233.25       |1616.5     |1617.5      |2024-01-02 07:50:08 |2024-01-02 11:29:29  |
|210.82       |500.0      |515.88      |

In [19]:
# ──── SQL Query 2: Top 10 Pickup Locations ────
print("📌 SQL Query 2: Top 10 Pickup Locations by Trip Count")
q2 = spark.sql("""
    SELECT PULocationID AS pickup_location,
           COUNT(*) AS trip_count,
           ROUND(AVG(fare_amount), 2) AS avg_fare,
           ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")
q2.show()

📌 SQL Query 2: Top 10 Pickup Locations by Trip Count
+---------------+----------+--------+------------+
|pickup_location|trip_count|avg_fare|avg_distance|
+---------------+----------+--------+------------+
|            132|    138130|   62.84|       15.94|
|            237|    137093|    12.3|         1.7|
|            161|    136500|   15.45|        2.32|
|            236|    129604|   12.72|        1.84|
|            162|    102308|   15.02|        2.25|
|            186|    100737|   16.05|        2.29|
|            230|    100008|   17.82|        2.96|
|            142|     98906|    13.5|         2.1|
|            138|     87253|   42.34|        9.71|
|            239|     82391|   13.34|        2.07|
+---------------+----------+--------+------------+



In [20]:
# ──── SQL Query 3: Average Fare by Payment Type ────
print("📌 SQL Query 3: Average Fare by Payment Type")
q3 = spark.sql("""
    SELECT payment_type,
           CASE payment_type
               WHEN 1 THEN 'Credit Card'
               WHEN 2 THEN 'Cash'
               WHEN 3 THEN 'No Charge'
               WHEN 4 THEN 'Dispute'
               WHEN 5 THEN 'Unknown'
               ELSE 'Other'
           END AS payment_name,
           COUNT(*) AS total_trips,
           ROUND(AVG(fare_amount), 2) AS avg_fare,
           ROUND(AVG(tip_amount), 2) AS avg_tip,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    GROUP BY payment_type
    ORDER BY total_trips DESC
""")
q3.show(truncate=False)

📌 SQL Query 3: Average Fare by Payment Type
+------------+------------+-----------+--------+-------+-------------+
|payment_type|payment_name|total_trips|avg_fare|avg_tip|total_revenue|
+------------+------------+-----------+--------+-------+-------------+
|1           |Credit Card |2298422    |18.38   |4.16   |6.452952225E7|
|2           |Cash        |422945     |18.61   |0.0    |1.008932501E7|
|4           |Dispute     |22879      |19.67   |0.0    |572064.61    |
|3           |No Charge   |10649      |17.04   |0.0    |235562.14    |
+------------+------------+-----------+--------+-------+-------------+



In [21]:
# ──── SQL Query 4: Peak Pickup Hour ────
print("📌 SQL Query 4: Peak Pickup Hours")
q4 = spark.sql("""
    SELECT HOUR(tpep_pickup_datetime) AS pickup_hour,
           COUNT(*) AS trip_count,
           ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM taxi_trips
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY trip_count DESC
""")
q4.show(24)

📌 SQL Query 4: Peak Pickup Hours
+-----------+----------+--------+
|pickup_hour|trip_count|avg_fare|
+-----------+----------+--------+
|         18|    198037|   16.94|
|         17|    193394|   18.04|
|         16|    180381|    19.4|
|         15|    179080|   19.04|
|         14|    173448|    19.2|
|         19|    172421|   17.62|
|         13|    161159|   18.36|
|         12|    155458|   17.75|
|         20|    150699|   18.06|
|         21|    149795|   18.35|
|         11|    142991|   17.57|
|         22|    131726|   19.15|
|         10|    131676|   18.02|
|          9|    120967|   17.84|
|          8|    106582|   17.57|
|         23|     98264|   20.37|
|          7|     75309|   18.54|
|          0|     69911|   19.69|
|          1|     45833|   17.35|
|          6|     35955|   21.99|
|          2|     32327|   16.35|
|          3|     20789|   17.87|
|          5|     15774|   27.54|
|          4|     12919|   22.97|
+-----------+----------+--------+



In [22]:
# ──── SQL Query 5: Trips Over 20 Miles ────
print("📌 SQL Query 5: Trips Over 20 Miles")
q5 = spark.sql("""
    SELECT COUNT(*) AS long_trips,
           ROUND(AVG(trip_distance), 2) AS avg_distance,
           ROUND(AVG(fare_amount), 2) AS avg_fare,
           ROUND(MIN(fare_amount), 2) AS min_fare,
           ROUND(MAX(fare_amount), 2) AS max_fare,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    WHERE trip_distance > 20
""")
q5.show()

📌 SQL Query 5: Trips Over 20 Miles
+----------+------------+--------+--------+--------+-------------+
|long_trips|avg_distance|avg_fare|min_fare|max_fare|total_revenue|
+----------+------------+--------+--------+--------+-------------+
|     28451|       25.22|    90.7|     0.0|  2221.3|   3260467.36|
+----------+------------+--------+--------+--------+-------------+



In [23]:
# ──── SQL Query 6: Daily Revenue (Monthly Revenue) ────
print("📌 SQL Query 6: Daily Revenue Summary")
q6 = spark.sql("""
    SELECT DATE(tpep_pickup_datetime) AS trip_date,
           COUNT(*) AS total_trips,
           ROUND(SUM(total_amount), 2) AS daily_revenue,
           ROUND(AVG(total_amount), 2) AS avg_trip_revenue
    FROM taxi_trips
    GROUP BY DATE(tpep_pickup_datetime)
    ORDER BY trip_date
""")
q6.show(31)

📌 SQL Query 6: Daily Revenue Summary
+----------+-----------+-------------+----------------+
| trip_date|total_trips|daily_revenue|avg_trip_revenue|
+----------+-----------+-------------+----------------+
|2002-12-31|          1|         10.5|            10.5|
|2009-01-01|          3|       127.69|           42.56|
|2023-12-31|         10|       224.62|           22.46|
|2024-01-01|      68127|   2123813.69|           31.17|
|2024-01-02|      71168|   2201717.23|           30.94|
|2024-01-03|      78229|   2287854.06|           29.25|
|2024-01-04|      97910|   2716359.02|           27.74|
|2024-01-05|      97524|   2628800.82|           26.96|
|2024-01-06|      89556|   2300717.43|           25.69|
|2024-01-07|      63053|   1816927.42|           28.82|
|2024-01-08|      75854|   2146958.86|            28.3|
|2024-01-09|      84510|   2183518.31|           25.84|
|2024-01-10|      89888|   2454074.39|            27.3|
|2024-01-11|      99230|   2793866.16|           28.16|
|2024-01-12

In [24]:
# ──── SQL Query 7: Average Tip Percentage by Payment Type ────
print("📌 SQL Query 7: Average Tip Percentage by Payment Type")
q7 = spark.sql("""
    SELECT payment_type,
           ROUND(AVG(tip_amount), 2) AS avg_tip,
           ROUND(AVG(CASE WHEN fare_amount > 0
                     THEN (tip_amount / fare_amount) * 100
                     ELSE 0 END), 2) AS avg_tip_pct
    FROM taxi_trips
    GROUP BY payment_type
    ORDER BY avg_tip_pct DESC
""")
q7.show()

📌 SQL Query 7: Average Tip Percentage by Payment Type
+------------+-------+-----------+
|payment_type|avg_tip|avg_tip_pct|
+------------+-------+-----------+
|           1|   4.16|      26.27|
|           3|    0.0|       0.04|
|           4|    0.0|       0.03|
|           2|    0.0|        0.0|
+------------+-------+-----------+



In [25]:
# ──── SQL Query 8: Trips by Passenger Count ────
print("📌 SQL Query 8: Trips by Passenger Count")
q8 = spark.sql("""
    SELECT CAST(passenger_count AS INT) AS passengers,
           COUNT(*) AS trip_count,
           ROUND(AVG(trip_distance), 2) AS avg_distance,
           ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM taxi_trips
    GROUP BY passenger_count
    ORDER BY passenger_count
""")
q8.show()

📌 SQL Query 8: Trips by Passenger Count
+----------+----------+------------+--------+
|passengers|trip_count|avg_distance|avg_fare|
+----------+----------+------------+--------+
|         0|     30680|        2.81|   16.49|
|         1|   2134966|        3.18|    17.9|
|         2|    395147|        3.83|   20.58|
|         3|     88975|         3.7|    20.4|
|         4|     49598|        3.99|   21.98|
|         5|     33308|        3.09|   17.59|
|         6|     22178|        2.97|   17.22|
|         7|         5|        3.67|   50.84|
|         8|        37|        2.14|    86.8|
|         9|         1|         1.8|    11.4|
+----------+----------+------------+--------+



In [26]:
# ──── SQL Query 9: Revenue by Vendor ────
print("📌 SQL Query 9: Revenue by Vendor")
q9 = spark.sql("""
    SELECT VendorID,
           COUNT(*) AS total_trips,
           ROUND(SUM(total_amount), 2) AS total_revenue,
           ROUND(AVG(total_amount), 2) AS avg_revenue_per_trip,
           ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY VendorID
    ORDER BY VendorID
""")
q9.show()

📌 SQL Query 9: Revenue by Vendor
+--------+-----------+-------------+--------------------+------------+
|VendorID|total_trips|total_revenue|avg_revenue_per_trip|avg_distance|
+--------+-----------+-------------+--------------------+------------+
|       1|     669555|1.732925144E7|               25.88|        3.13|
|       2|    2085340|5.809722257E7|               27.86|        3.35|
+--------+-----------+-------------+--------------------+------------+



In [27]:
# ──── SQL Query 10: Trips by Rate Code ────
print("📌 SQL Query 10: Trips by Rate Code")
q10 = spark.sql("""
    SELECT RatecodeID,
           CASE CAST(RatecodeID AS INT)
               WHEN 1 THEN 'Standard Rate'
               WHEN 2 THEN 'JFK'
               WHEN 3 THEN 'Newark'
               WHEN 4 THEN 'Nassau/Westchester'
               WHEN 5 THEN 'Negotiated Fare'
               WHEN 6 THEN 'Group Ride'
               ELSE 'Unknown'
           END AS rate_name,
           COUNT(*) AS trip_count,
           ROUND(AVG(fare_amount), 2) AS avg_fare,
           ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY RatecodeID
    ORDER BY trip_count DESC
""")
q10.show()

📌 SQL Query 10: Trips by Rate Code
+----------+------------------+----------+--------+------------+
|RatecodeID|         rate_name|trip_count|avg_fare|avg_distance|
+----------+------------------+----------+--------+------------+
|         1|     Standard Rate|   2611299|   15.78|        2.61|
|         2|               JFK|     94196|   69.96|       18.02|
|        99|           Unknown|     27142|   33.51|        8.27|
|         5|   Negotiated Fare|      8928|   81.11|        9.95|
|         3|            Newark|      7214|   88.23|       17.41|
|         4|Nassau/Westchester|      6115|  109.44|       23.17|
|         6|        Group Ride|         1|     2.5|        4.45|
+----------+------------------+----------+--------+------------+



In [28]:
# ──── Save Query Outputs to CSV ────
print("💾 Saving query results to output/ folder...")

q1.toPandas().to_csv("output/query1.csv", index=False)
print("   ✅ output/query1.csv — Top 10 longest trips")

q2.toPandas().to_csv("output/query2.csv", index=False)
print("   ✅ output/query2.csv — Top pickup locations")

q3.toPandas().to_csv("output/query3.csv", index=False)
print("   ✅ output/query3.csv — Average fare by payment type")

print("\n✅ All query outputs saved!")

💾 Saving query results to output/ folder...
   ✅ output/query1.csv — Top 10 longest trips
   ✅ output/query2.csv — Top pickup locations
   ✅ output/query3.csv — Average fare by payment type

✅ All query outputs saved!


---
## Part 7: Window Functions (10 Marks)
Using: row_number(), rank(), dense_rank()



In [29]:
# Window spec: partition by pickup location, order by fare descending
windowSpec = Window.partitionBy("PULocationID").orderBy(col("fare_amount").desc())

In [30]:
# ──── Window Function 1: row_number() ────
print("📌 Window Function 1: row_number()")
print("   Purpose: Assign unique sequential numbers to trips")
print("   within each pickup location, ordered by fare (highest first)\n")
df_rownum = df_clean.withColumn("row_num", row_number().over(windowSpec))
df_rownum.select("PULocationID", "fare_amount", "trip_distance", "row_num") \
    .filter(col("row_num") <= 3) \
    .orderBy("PULocationID", "row_num") \
    .show(15)

📌 Window Function 1: row_number()
   Purpose: Assign unique sequential numbers to trips
   within each pickup location, ordered by fare (highest first)

+------------+-----------+-------------+-------+
|PULocationID|fare_amount|trip_distance|row_num|
+------------+-----------+-------------+-------+
|           1|      155.0|         0.17|      1|
|           1|      150.0|          0.3|      2|
|           1|      150.0|         0.01|      3|
|           2|       70.0|        18.69|      1|
|           2|       36.6|          9.6|      2|
|           3|      216.5|         35.7|      1|
|           3|       80.0|          1.7|      2|
|           3|       73.5|         26.2|      3|
|           4|      276.0|        30.67|      1|
|           4|      160.0|        46.98|      2|
|           4|      84.19|         0.02|      3|
|           6|      160.0|         12.6|      1|
|           6|       76.9|        15.42|      2|
|           6|       75.0|         15.7|      3|
|           7|

In [31]:
# ──── Window Function 2: rank() ────
print("📌 Window Function 2: rank()")
print("   Purpose: Rank trips by fare within each pickup location")
print("   (ties get the same rank, then ranks are skipped)\n")
df_rank = df_clean.withColumn("fare_rank", rank().over(windowSpec))
df_rank.select("PULocationID", "fare_amount", "trip_distance", "fare_rank") \
    .filter(col("fare_rank") <= 3) \
    .orderBy("PULocationID", "fare_rank") \
    .show(15)

📌 Window Function 2: rank()
   Purpose: Rank trips by fare within each pickup location
   (ties get the same rank, then ranks are skipped)

+------------+-----------+-------------+---------+
|PULocationID|fare_amount|trip_distance|fare_rank|
+------------+-----------+-------------+---------+
|           1|      155.0|         0.17|        1|
|           1|      150.0|          0.3|        2|
|           1|      150.0|         0.01|        2|
|           2|       70.0|        18.69|        1|
|           2|       36.6|          9.6|        2|
|           3|      216.5|         35.7|        1|
|           3|       80.0|          1.7|        2|
|           3|       73.5|         26.2|        3|
|           4|      276.0|        30.67|        1|
|           4|      160.0|        46.98|        2|
|           4|      84.19|         0.02|        3|
|           6|      160.0|         12.6|        1|
|           6|       76.9|        15.42|        2|
|           6|       75.0|         15.7|    

In [32]:
# ──── Window Function 3: dense_rank() ────
print("📌 Window Function 3: dense_rank()")
print("   Purpose: Dense rank trips — no gaps in ranking for ties\n")
df_dense = df_clean.withColumn("dense_fare_rank", dense_rank().over(windowSpec))
df_dense.select("PULocationID", "fare_amount", "trip_distance", "dense_fare_rank") \
    .filter(col("dense_fare_rank") <= 3) \
    .orderBy("PULocationID", "dense_fare_rank") \
    .show(15)

📌 Window Function 3: dense_rank()
   Purpose: Dense rank trips — no gaps in ranking for ties

+------------+-----------+-------------+---------------+
|PULocationID|fare_amount|trip_distance|dense_fare_rank|
+------------+-----------+-------------+---------------+
|           1|      155.0|         0.17|              1|
|           1|      150.0|          0.3|              2|
|           1|      150.0|         0.01|              2|
|           1|     148.55|         0.01|              3|
|           2|       70.0|        18.69|              1|
|           2|       36.6|          9.6|              2|
|           3|      216.5|         35.7|              1|
|           3|       80.0|          1.7|              2|
|           3|       73.5|         26.2|              3|
|           4|      276.0|        30.67|              1|
|           4|      160.0|        46.98|              2|
|           4|      84.19|         0.02|              3|
|           6|      160.0|         12.6|           

---
## Part 8: Performance Optimization (10 Marks)
Demonstrate: cache(), repartition(), explain(), and execution time comparison



In [33]:

# Uncache if previously cached
df_clean.unpersist()

# ──── Test WITHOUT cache ────
print("🔴 Test 1: WITHOUT Cache")
start_time = time.time()
result1 = df_clean.groupBy("PULocationID").agg(
    count("*").alias("trips"),
    avg("fare_amount").alias("avg_fare")
).collect()
no_cache_time = time.time() - start_time
print(f"   Execution Time: {no_cache_time:.4f} seconds")

🔴 Test 1: WITHOUT Cache
   Execution Time: 10.4956 seconds


In [34]:
# ──── Test WITH cache ────
print("🟢 Test 2: WITH Cache")
df_clean.cache()
df_clean.count()  # trigger caching

start_time = time.time()
result2 = df_clean.groupBy("PULocationID").agg(
    count("*").alias("trips"),
    avg("fare_amount").alias("avg_fare")
).collect()
cache_time = time.time() - start_time
print(f"   Execution Time: {cache_time:.4f} seconds")

print(f"\n{'=' * 60}")
print(f"📊 PERFORMANCE COMPARISON:")
print(f"   Without Cache : {no_cache_time:.4f} seconds")
print(f"   With Cache    : {cache_time:.4f} seconds")
if cache_time > 0:
    print(f"   Speedup       : {no_cache_time/cache_time:.2f}x faster")
print(f"{'=' * 60}")

🟢 Test 2: WITH Cache
   Execution Time: 0.8985 seconds

📊 PERFORMANCE COMPARISON:
   Without Cache : 10.4956 seconds
   With Cache    : 0.8985 seconds
   Speedup       : 11.68x faster


In [37]:
# ──── Repartition Demo ────
print("\n📌 Repartition Demonstration:")
print(f"   Current Partitions : {df_clean.rdd.getNumPartitions()}")
df_repart = df_clean.repartition(8)
print(f"   After repartition(8): {df_repart.rdd.getNumPartitions()}")
df_coalesce = df_clean.coalesce(2)
print(f"   After coalesce(2)   : {df_coalesce.rdd.getNumPartitions()}")


📌 Repartition Demonstration:
   Current Partitions : 3
   After repartition(8): 8
   After coalesce(2)   : 2


In [39]:
# ──── Execution Plan (explain) ────
print("📌 Execution Plan — explain(True):")
print("   Query: GroupBy PULocationID → Average fare_amount\n")
df_clean.groupBy("PULocationID").agg(avg("fare_amount")).explain(True)

📌 Execution Plan — explain(True):
   Query: GroupBy PULocationID → Average fare_amount

== Parsed Logical Plan ==
'Aggregate ['PULocationID], ['PULocationID, avg('fare_amount) AS avg(fare_amount)#5108]
+- Filter atleastnnonnulls(7, VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, fare_amount#10, total_amount#16)
   +- Filter (fare_amount#10 >= cast(0 as double))
      +- Filter (trip_distance#4 > cast(0 as double))
         +- Deduplicate [DOLocationID#8, improvement_surcharge#15, tpep_dropoff_datetime#2, PULocationID#7, trip_distance#4, Airport_fee#18, tolls_amount#14, RatecodeID#5L, VendorID#0, tip_amount#13, payment_type#9L, fare_amount#10, passenger_count#3L, store_and_fwd_flag#6, extra#11, congestion_surcharge#17, total_amount#16, tpep_pickup_datetime#1, mta_tax#12]
            +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,D

---
## Part 9: Spark UI Analysis (5 Marks)
In Google Colab, we access the Spark UI via Colab's built-in port proxy.



In [45]:
# This cell opens the Spark UI in Colab.
# Take screenshots of Jobs, Stages, Storage, Executors tabs.

!npm install -g localtunnel
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
!lt --port 4040


⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 928ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Password/Endpoint IP for localtunnel is: 34.81.32.67
your url is: https://tiny-sites-sleep.loca.lt
^C


In [48]:
# ──── Download Output Files ────
from google.colab import files
import shutil

# Download CSV outputs
print("📥 Downloading output files...")
shutil.make_archive("output", "zip", "output")
files.download("output.zip")
print("✅ output.zip downloaded!")

📥 Downloading output files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ output.zip downloaded!


In [49]:
# ──── Stop Spark Session ────
print("\n🛑 Stopping Spark Session...")
spark.stop()
print("✅ Spark Session stopped successfully.")
print("\n🎉 Assignment Complete! All parts executed successfully.")


🛑 Stopping Spark Session...
✅ Spark Session stopped successfully.

🎉 Assignment Complete! All parts executed successfully.


In [50]:
# ──── Download the notebook ────
# In Colab: File → Download → Download .ipynb
# Rename the downloaded file to: notebook.ipynb
print("📥 To download your notebook:")
print("   Go to: File → Download → Download .ipynb")
print("   Rename it to: notebook.ipynb")

📥 To download your notebook:
   Go to: File → Download → Download .ipynb
   Rename it to: notebook.ipynb
